# Mini Project 1: AI Text Summarizer
**Skill Nexis | LLM Development | Week 1**

**Model:** `sshleifer/distilbart-cnn-12-6` (a distilled BART model fine-tuned for news summarization), loaded with the Hugging Face `transformers` library.

**Features**
- Accepts text input and returns a short summary
- Supports a requested summary length (`short`, `medium`, `long`, or a target word count)
- Handles empty and excessively long input
- Evaluates five examples for factual accuracy


In [1]:
# Run once (Colab / fresh environment)
!pip install -q transformers torch pandas

In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch
import re
import pandas as pd

MODEL_NAME = "sshleifer/distilbart-cnn-12-6"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device).eval()
print("Model loaded on", device)

config.json:   0%|          | 0.00/1.80k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 1.22GB            

pytorch_model.bin: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


model.safetensors: reconstructing file:   0%|          |  0.00B / 1.22GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

Model loaded on cpu


## 1. Summarizer function

**Design choices**
- **Length control:** the user picks a preset or a word count, which is converted to `min_length` / `max_length` in tokens (about 1.6 tokens per word).
- **Empty input:** rejected with a clear error message.
- **Too-short input:** returned as is, since summarizing a one-liner only adds risk of errors.
- **Excessively long input:** the model reads about 1,024 tokens at once, so long text is split into sentence-based chunks, each chunk is summarized, and the partial summaries are summarized again (map-reduce). Input above `MAX_INPUT_WORDS` is rejected.


In [3]:
LENGTH_PRESETS = {"short": 30, "medium": 60, "long": 100}   # target words
MAX_INPUT_WORDS = 20000     # hard limit: reject anything bigger
MIN_INPUT_WORDS = 30        # below this, summarizing is pointless
CHUNK_TOKENS = 700          # safe chunk size for a 1024-token model

def _word_count(text):
    return len(text.split())

def _split_chunks(text, max_tokens=CHUNK_TOKENS):
    """Split text into sentence-aligned chunks under max_tokens."""
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    chunks, current, current_len = [], [], 0
    for s in sentences:
        n = len(tokenizer.encode(s, add_special_tokens=False))
        if current and current_len + n > max_tokens:
            chunks.append(" ".join(current))
            current, current_len = [], 0
        current.append(s)
        current_len += n
    if current:
        chunks.append(" ".join(current))
    return chunks

def clean_summary(summary):
    """Fix spacing before punctuation and drop a trailing incomplete sentence."""
    s = re.sub(r"\s+([.,;:!?])", r"\1", summary.strip())
    if s and s[-1] not in ".!?":
        ends = list(re.finditer(r"[.!?](?=\s|$)", s))
        if ends:
            s = s[:ends[-1].end()]
    return s

def _run_model(text, target_words):
    max_len = int(target_words * 1.6)
    min_len = max(10, int(max_len * 0.5))
    in_len = len(tokenizer.encode(text, add_special_tokens=False))
    max_len = min(max_len, max(12, in_len - 1))   # never longer than the input
    min_len = min(min_len, max_len - 1)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=1024).to(device)
    with torch.no_grad():
        ids = model.generate(**inputs, max_length=max_len, min_length=min_len,
                             num_beams=4, do_sample=False,
                             no_repeat_ngram_size=3, early_stopping=True)
    return clean_summary(tokenizer.decode(ids[0], skip_special_tokens=True))

def summarize(text, length="medium"):
    """Summarize `text`. `length` is 'short' | 'medium' | 'long' or an int (target words)."""
    # --- input validation ---
    if text is None or not text.strip():
        raise ValueError("Input is empty. Please provide some text to summarize.")
    words = _word_count(text)
    if words > MAX_INPUT_WORDS:
        raise ValueError(f"Input too long ({words} words). Limit is {MAX_INPUT_WORDS}.")
    if words < MIN_INPUT_WORDS:
        return text.strip()   # already short enough

    # --- resolve requested length ---
    if isinstance(length, str):
        if length not in LENGTH_PRESETS:
            raise ValueError(f"length must be one of {list(LENGTH_PRESETS)} or an int.")
        target = LENGTH_PRESETS[length]
    else:
        target = max(15, int(length))

    # --- summarize (map-reduce for long input) ---
    chunks = _split_chunks(text)
    if len(chunks) == 1:
        return _run_model(chunks[0], target)
    partials = [_run_model(c, max(40, target)) for c in chunks]
    combined = " ".join(partials)
    return _run_model(combined, target)

## 2. Edge-case tests

In [4]:
tests = {
    "empty string": "",
    "whitespace only": "   \n  ",
    "too short": "The meeting is at 3 PM tomorrow.",
    "too long": "word " * (MAX_INPUT_WORDS + 5),
}
for name, t in tests.items():
    try:
        print(f"[{name}] ->", summarize(t))
    except ValueError as e:
        print(f"[{name}] -> ValueError: {e}")

[empty string] -> ValueError: Input is empty. Please provide some text to summarize.
[whitespace only] -> ValueError: Input is empty. Please provide some text to summarize.
[too short] -> The meeting is at 3 PM tomorrow.
[too long] -> ValueError: Input too long (20005 words). Limit is 20000.


## 3. Five test examples
The passages below are short original texts written for this notebook, with a mix of numbers, names, dates and events so that factual errors are easy to spot.


In [5]:
examples = [
 {"id": 1, "topic": "Science",
  "text": """Researchers at a university marine lab reported that a coral reef off the coast of Indonesia recovered about 60 percent of its lost coverage within six years after a severe bleaching event in 2016. The team, led by Dr. Ananya Rao, attributed the recovery to a ban on fishing in the area and to unusually stable water temperatures between 2018 and 2022. They surveyed 40 sites by diving and by using underwater drones. However, the researchers warned that the reef remains vulnerable, since a single marine heatwave could undo much of the progress. The study was published in a peer-reviewed journal in March and recommended extending the fishing ban to neighbouring reefs."""},
 {"id": 2, "topic": "Business",
  "text": """A mid-sized electronics retailer announced that it will close 25 of its 140 stores by the end of next year, citing a shift toward online shopping and rising rents. The company said that about 900 employees would be affected, and that roughly 300 of them would be offered roles at nearby stores. Chief executive Marcus Lindqvist said the savings, estimated at 18 million dollars annually, would be invested in its delivery network and mobile app. Shares of the company fell 4 percent on the news. Analysts noted that the retailer's online sales grew 22 percent last year, but that its overall profit margin fell to 3 percent."""},
 {"id": 3, "topic": "Technology",
  "text": """A new open-source programming tool released this week lets developers run small language models directly in a web browser without sending data to a server. The project, maintained by a group of volunteers, supports models of up to about 3 billion parameters and uses the graphics processor of the user's device to speed up processing. Early testers reported that a typical response took two to five seconds on a modern laptop, but that older phones struggled and sometimes ran out of memory. The developers said privacy was the main motivation, because no text ever leaves the user's machine. They plan to add support for voice input in the next version."""},
 {"id": 4, "topic": "Health",
  "text": """A city health department launched a campaign to increase flu vaccination among adults over 65 after last winter's rates dropped to 58 percent, down from 67 percent two years earlier. The campaign includes free vaccination at 12 community pharmacies, mobile clinics visiting senior centers every Wednesday, and reminder calls from volunteers. Health officials said that older adults are far more likely to be hospitalized with flu complications. The department set a goal of reaching 70 percent coverage by the end of March. Funding of 450,000 dollars comes from a state grant, and the program will be reviewed after one year."""},
 {"id": 5, "topic": "Education",
  "text": """A school district in the northern part of the state will move to a four-day school week beginning in September, with classes ending at 3:45 PM Monday through Thursday and Fridays reserved for optional tutoring and activities. District leaders said the change would save about 1.2 million dollars in transportation and energy costs and help recruit teachers, since the district has 35 unfilled positions. Some parents expressed concern about childcare on Fridays, so the district will offer low-cost supervised programs at three elementary schools. A review of test scores will be carried out after two years, and the board can reverse the decision if results decline."""},
]
for e in examples:
    print(e["id"], e["topic"], "-", _word_count(e["text"]), "words")

1 Science - 112 words
2 Business - 107 words
3 Technology - 110 words
4 Health - 100 words
5 Education - 106 words


In [15]:
results = []
for e in examples:
    summ = summarize(e["text"], length="short")
    results.append({"id": e["id"], "topic": e["topic"], "summary": summ,
                    "summary_words": _word_count(summ)})
    print(f"--- Example {e['id']} ({e['topic']}) ---")
    print(summ, "\n")


--- Example 1: Science ---
Summary: A coral reef off the coast of Indonesia recovered about 60 percent of its lost coverage within six years after a severe bleaching event in 2016.
Words: 26

--- Example 2: Business ---
Summary: Mid-sized electronics retailer announced that it will close 25 of its 140 stores by the end of next year.
Words: 19

--- Example 3: Technology ---
Summary: A new programming tool lets developers run small language models directly in a web browser without sending data to a server.
Words: 21

--- Example 4: Health ---
Summary: Last winter's rates dropped to 58 percent, down from 67 percent two years earlier. The department set a goal of reaching 70 percent coverage by the end of March. The program will be reviewed after one year.
Words: 37

--- Example 5: Education ---
Summary: A school district in the northern part of the state will move to a four-day school week beginning in September. Classes ending at 3:45 PM Monday through Thursday and Fridays reserved for 

In [16]:
for e in examples:
    print(e["id"], e["topic"], "-", _word_count(e["text"]), "words")

1 Science - 112 words
2 Business - 107 words
3 Technology - 110 words
4 Health - 100 words
5 Education - 106 words


In [18]:

results = []

for e in examples:
    summ = summarize(e["text"], length="short")

    results.append({
        "id": e["id"],
        "topic": e["topic"],
        "summary": summ,
        "summary_words": _word_count(summ)
    })

    print(f"\n--- Example {e['id']}: {e['topic']} ---")
    print("Summary:", summ)
    print("Words:", _word_count(summ))



--- Example 1: Science ---
Summary: A coral reef off the coast of Indonesia recovered about 60 percent of its lost coverage within six years after a severe bleaching event in 2016.
Words: 26

--- Example 2: Business ---
Summary: Mid-sized electronics retailer announced that it will close 25 of its 140 stores by the end of next year.
Words: 19

--- Example 3: Technology ---
Summary: A new programming tool lets developers run small language models directly in a web browser without sending data to a server.
Words: 21

--- Example 4: Health ---
Summary: Last winter's rates dropped to 58 percent, down from 67 percent two years earlier. The department set a goal of reaching 70 percent coverage by the end of March. The program will be reviewed after one year.
Words: 37

--- Example 5: Education ---
Summary: A school district in the northern part of the state will move to a four-day school week beginning in September. Classes ending at 3:45 PM Monday through Thursday and Fridays reserved for 

## 4. Length control demo

In [7]:
for L in ["short", "medium", "long", 25]:
    s = summarize(examples[1]["text"], length=L)
    print(f"length={L!s:7} -> {_word_count(s):3} words | {s}\n")

length=short   ->  19 words | Mid-sized electronics retailer announced that it will close 25 of its 140 stores by the end of next year.

length=medium  ->  64 words | Mid-sized electronics retailer announced that it will close 25 of its 140 stores by the end of next year. The company said that about 900 employees would be affected, and that roughly 300 of them would be offered roles at nearby stores. Analysts noted that the retailer's online sales grew 22 percent last year, but that its overall profit margin fell to 3 percent.

length=long    ->  74 words | Mid-sized electronics retailer announced that it will close 25 of its 140 stores by the end of next year. The company said that about 900 employees would be affected, and that roughly 300 of them would be offered roles at nearby stores. Analysts noted that the retailer's online sales grew 22 percent last year, but that its overall profit margin fell to 3 percent. Shares of the company fell 4 percent on the news.

length=25      ->  

## 5. Evaluation: factual accuracy

Two checks are used:

1. **Automatic check:** every number and capitalized name in the summary must also appear in the source text. Anything that doesn't is flagged as a possible hallucination.
2. **Manual check:** read each summary against its source and fill in the table in the next cell.


In [8]:
def extract_facts(text):
    nums = set(re.findall(r"\d[\d,\.]*", text))
    nums = {n.rstrip(".,") for n in nums}
    caps = set(re.findall(r"(?<![.!?]\s)(?<!^)\b[A-Z][a-z]+", text))
    return nums, caps

def auto_check(source, summary):
    s_nums, s_caps = extract_facts(summary)
    src = source.replace(",", "")
    unsupported = [n for n in s_nums if n.replace(",", "") not in src]
    unsupported += [c for c in s_caps if c not in source]
    return unsupported

rows = []
for e, r in zip(examples, results):
    unsup = auto_check(e["text"], r["summary"])
    rows.append({"id": r["id"], "topic": r["topic"],
                 "summary_words": r["summary_words"],
                 "unsupported_facts": unsup if unsup else "none",
                 "auto_flag": "CHECK" if unsup else "OK"})
auto_df = pd.DataFrame(rows)
auto_df

,id,topic,summary_words,unsupported_facts,auto_flag
0,1,Science,26,none,OK
1,2,Business,19,none,OK
2,3,Technology,21,none,OK
3,4,Health,37,none,OK
4,5,Education,36,none,OK


### Manual evaluation
After reading each summary, replace the `TODO` values below. Use: **Accurate** (all claims supported), **Minor issue** (small omission or imprecise wording), **Inaccurate** (contains an unsupported or wrong claim).


In [19]:

manual = pd.DataFrame([
    {
        "id": 1,
        "verdict": "Minor issue",
        "key_facts_kept": "Indonesia coral reef, 60% recovery, six years, 2016 bleaching event",
        "errors_or_notes": "Causes of recovery and vulnerability warning omitted"
    },
    {
        "id": 2,
        "verdict": "Minor issue",
        "key_facts_kept": "25 of 140 stores closing by end of next year",
        "errors_or_notes": "Employee impact, savings and financial details omitted"
    },
    {
        "id": 3,
        "verdict": "Minor issue",
        "key_facts_kept": "Browser-based language models without sending data to a server",
        "errors_or_notes": "Performance, device limitations and future voice support omitted"
    },
    {
        "id": 4,
        "verdict": "Minor issue",
        "key_facts_kept": "Vaccination rates 58%, previous 67%, target 70%",
        "errors_or_notes": "Campaign activities, funding and mobile clinics omitted"
    },
    {
        "id": 5,
        "verdict": "Minor issue",
        "key_facts_kept": "Four-day school week, September start, school timings",
        "errors_or_notes": "Cost savings, teacher vacancies and childcare concerns omitted; second sentence is incomplete"
    }
])

final = auto_df.merge(manual, on="id")
final


,id,topic,summary_words,unsupported_facts,auto_flag,verdict,key_facts_kept,errors_or_notes
0,1,Science,26,none,OK,Minor issue,"Indonesia coral reef, 60% recovery, six years,...",Causes of recovery and vulnerability warning o...
1,2,Business,19,none,OK,Minor issue,25 of 140 stores closing by end of next year,"Employee impact, savings and financial details..."
2,3,Technology,21,none,OK,Minor issue,Browser-based language models without sending ...,"Performance, device limitations and future voi..."
3,4,Health,37,none,OK,Minor issue,"Vaccination rates 58%, previous 67%, target 70%","Campaign activities, funding and mobile clinic..."
4,5,Education,36,none,OK,Minor issue,"Four-day school week, September start, school ...","Cost savings, teacher vacancies and childcare ..."


## 6. Model and prompt explanation

**Model.** `distilbart-cnn-12-6` is an *extractive-abstractive* style seq2seq model: an encoder reads the passage and a decoder writes a new summary token by token. It is a smaller, faster version of BART fine-tuned on CNN/DailyMail news.

**Prompts.** This model is a task-specific summarizer, so it does not take a natural-language instruction like a chat LLM. The "prompt" is the input text plus decoding controls: `min_length`/`max_length` (set from the requested summary length), beam search (`num_beams=4`) and `do_sample=False` (deterministic output, which suits factual tasks).

**Handling long input.** The model reads about 1,024 tokens, so long text is chunked by sentence, each chunk is summarized, and the partial summaries are summarized again.

**Limitations.**
- Trained on news, so style and quality may drop on other genres (legal, technical, dialogue).
- Abstractive models can paraphrase numbers or merge facts incorrectly, which is why the evaluation checks numbers and names.
- The automatic check only catches facts missing from the source. It cannot catch a correct fact used in a misleading way, so the manual review is needed.

**Possible improvements.** Try a larger model (`facebook/bart-large-cnn`), or an LLM API with an explicit instruction prompt such as *"Summarize in 2 sentences. Use only facts in the text."*
